In [ ]:
print("GPU Information:")

!nvidia-smi

In [ ]:
import os
from google.colab import drive

# ================= MOUNT DRIVE =================
drive.mount('/content/drive')

# ================= CONFIGURATION =================
BASE_PATH = "/content/drive/MyDrive/sparkyllm/datasets_pretrain"
OUTPUT_FILE = os.path.join(BASE_PATH, "training_data_long.txt")
SEPARATOR = "\n<|endoftext|>\n"

# ================= HELPERS =================
def read_txt_files(directory):
    texts = []
    if not os.path.exists(directory):
        print(f"  Directory not found: {directory}")
        return texts
    for filename in sorted(os.listdir(directory)):
        if filename.endswith(".txt"):
            path = os.path.join(directory, filename)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                    content = content.replace("<|para|>", "\n\n")
                    texts.append(content)
                    print(f"  {filename}")
            except Exception as e:
                print(f"  Skipped {filename}: {e}")
    return texts

# ================= SCAN ALL data_* FOLDERS =================
texts = []
data_dirs = sorted([
    d for d in os.listdir(BASE_PATH)
    if d.startswith("data_") and os.path.isdir(os.path.join(BASE_PATH, d))
])

if not data_dirs:
    print(f"No data_* folders found in {BASE_PATH}")
else:
    for dirname in data_dirs:
        dirpath = os.path.join(BASE_PATH, dirname)
        print(f"\n[{dirname}]")
        dir_texts = read_txt_files(dirpath)
        texts.extend(dir_texts)
        print(f"  -> {len(dir_texts)} files")

merged = SEPARATOR.join(texts)

# ================= SAVE FILE =================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(merged)

# ================= SUMMARY =================
print(f"\nDone!")
print(f"  Sources: {len(data_dirs)} folders, {len(texts)} files")
print(f"  Saved: {OUTPUT_FILE} ({len(merged):,} chars)")

In [ ]:
# ================= SAVE MANIFEST =================
import hashlib, json, datetime

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
MANIFEST_DIR = os.path.join(SPARKYLLM, "manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def file_info(path):
    if not os.path.exists(path):
        return {"path": path, "exists": False}
    return {
        "path": path,
        "size_mb": round(os.path.getsize(path) / 1024 / 1024, 2),
        "sha256": file_hash(path),
    }

# Hash every source file, grouped by data_* folder
all_sources = {}
for dirname in data_dirs:
    dirpath = os.path.join(BASE_PATH, dirname)
    files = []
    for filename in sorted(os.listdir(dirpath)):
        if filename.endswith(".txt"):
            files.append({"filename": filename, **file_info(os.path.join(dirpath, filename))})
    all_sources[dirname] = files

# Single hash over all source hashes for quick comparison
all_hashes = "".join(
    f["sha256"] for files in all_sources.values() for f in files if "sha256" in f
)
combined_hash = hashlib.sha256(all_hashes.encode()).hexdigest()[:16]

manifest = {
    "stage": "consolidation",
    "created": datetime.datetime.now().isoformat(),
    "sources_hash": combined_hash,
    "num_data_dirs": len(data_dirs),
    "num_source_files": sum(len(files) for files in all_sources.values()),
    "data_dirs": {dirname: files for dirname, files in all_sources.items()},
    "output": file_info(OUTPUT_FILE),
    "separator": SEPARATOR,
    "preprocessing": ["<|para|> -> \\n\\n"],
}

manifest_path = os.path.join(MANIFEST_DIR, "consolidation_latest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nManifest saved: {manifest_path}")
print(f"  sources_hash: {combined_hash}")
print(f"  {manifest['num_data_dirs']} folders, {manifest['num_source_files']} files")